# Verifikasi kandidat SENA v0.7 dan regresi NARA

Ekspor setelah pelatihan selesai. Kandidat terikat hash bobot, konfigurasi v0.4, dan lock v0.7. Target dan cutoff 0.95 tetap. Tidak ada parameter fitting dalam notebook verifikasi ini.

Development manusia146 serta asisten74 yang sudah dilihat dipakai memilih model. Regresi manusia50 lama bukan penerimaan independen. Diagnostik50 baru v0.7 dibekukan sebelum pelatihan, diperiksa sesudah pemilihan, dan tidak dipakai mengubah kandidat. Label diagnostik adalah usulan asisten, belum acuan manusia. Persetujuan mutu dan Done tetap memerlukan proses penerimaan manusia.

In [1]:
from pathlib import Path
import os,sys,json,subprocess
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT'])
WORK=Path(os.environ['SLR09_WORK_DIR'])
RUN=WORK/'training_matched_v07'
OUTPUT=WORK/'quality_diagnostics_v07'
def execute(command):
 p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 for line in p.stdout: print(line,end='',flush=True)
 p.wait()
 if p.returncode: raise RuntimeError('Step failed: '+str(p.returncode))
assert json.loads((RUN/'run_state.json').read_text())['status']=='SUCCEEDED'

In [2]:
execute([sys.executable,str(ROOT/'code/quality_fix/prepare_matched_candidate_v07.py'),
 '--run',str(RUN),'--output',str(WORK/'exports')])

{
  "archive": "/workspace/scratch/e5625dd03138/silar09-next-v06/exports/SILAR_SENA_matched_v0.7_20261019.zip",
  "bytes": 411889984,
  "sha256": "2965cd389299e56f8a51816a7012b92feeb4767b73367a9896b2a932942ac76e",
  "model_id": "SILAR_SENA_matched_v07_20261019T054716",
  "selected_epoch": 1,
  "selected_bias": 0.1,
  "quality_acceptance_approved": false
}


In [3]:
execute([sys.executable,str(ROOT/'code/run_active_tests.py'),
 '--evidence-dir',str(WORK/'contract_tests_v07')])

test_explicit_breathing_condition_is_kept (test_candidate_runtime.ActiveNormalizationTests.test_explicit_breathing_condition_is_kept) ... ok
test_literal_impact_still_maps_to_impact (test_candidate_runtime.ActiveNormalizationTests.test_literal_impact_still_maps_to_impact) ... ok
test_negated_breathing_condition_is_not_positive_victim (test_candidate_runtime.ActiveNormalizationTests.test_negated_breathing_condition_is_not_positive_victim) ... ok
test_unconscious_condition_is_not_misread_as_negation (test_candidate_runtime.ActiveNormalizationTests.test_unconscious_condition_is_not_misread_as_negation) ... ok
test_candidate_directory_has_explicit_identity_and_hashes (test_candidate_runtime.CandidateAssetTests.test_candidate_directory_has_explicit_identity_and_hashes) ... ok
test_config_labels_must_match_even_if_hash_was_updated (test_candidate_runtime.CandidateAssetTests.test_config_labels_must_match_even_if_hash_was_updated) ... ok
test_different_weights_cannot_be_accepted_as_candidate (

In [4]:
execute([sys.executable,str(ROOT/'code/quality_fix/run_matched_diagnostics_v07.py'),
 '--previous',str(ROOT/'evidence/quality_fix_v05/previous_50'),
 '--seen',str(ROOT/'data/mvp_seen_development_v07.json'),
 '--fresh',str(ROOT/'data/frozen_new_diagnostic_v0.7.json'),
 '--sena-dir',str(RUN/'best_model'),
 '--nara-archive',str(WORK/'assets/CT/SILAR_NARA_reviewed_pilot_20261018T001122_607337Z.zip'),
 '--training-run',str(RUN),'--output',str(OUTPUT)])

Old pilot 10 / 50
Old pilot 20 / 50
Old pilot 30 / 50
Old pilot 40 / 50
Old pilot 50 / 50
seen74 10 / 74
seen74 20 / 74
seen74 30 / 74
seen74 40 / 74
seen74 50 / 74
seen74 60 / 74
seen74 70 / 74
new50 10 / 50
new50 20 / 50
new50 30 / 50
new50 40 / 50
new50 50 / 50
{
  "old50_targets_met": false,
  "seen74": {
    "accuracy": 0.972972972972973,
    "macro_f1": 0.9755016835016835,
    "non_event_recall": 0.9545454545454546,
    "minimum_event_recall": 0.9230769230769231,
    "targets_met": true,
    "errors": 2
  },
  "new50": {
    "accuracy": 0.96,
    "macro_f1": 0.9599498746867168,
    "non_event_recall": 0.9,
    "minimum_event_recall": 0.9,
    "targets_met": true,
    "errors": 2
  },
  "quality_acceptance_approved": false
}
{
  "schema": "slr09-matched-diagnostics-0.7",
  "status": "SUCCEEDED",
  "started_at": "2026-10-19T05:54:29.664346+00:00",
  "training_executed": false,
  "parameter_fitting_executed": false,
  "model_inference_executed": true,
  "quality_acceptance_approved"

In [5]:
state=json.loads((OUTPUT/'run_state.json').read_text())
assert state['status']=='SUCCEEDED' and state['model_inference_executed']
assert not state['parameter_fitting_executed'] and not state['quality_acceptance_approved']
print(json.dumps(state['diagnostic_summary'],indent=2))
print('Human acceptance remains pending; no production deployment.')

{
  "seen74": {
    "accuracy": 0.972972972972973,
    "macro_f1": 0.9755016835016835,
    "non_event_recall": 0.9545454545454546,
    "minimum_event_recall": 0.9230769230769231,
    "targets_met": true,
    "errors": 2
  },
  "new50": {
    "accuracy": 0.96,
    "macro_f1": 0.9599498746867168,
    "non_event_recall": 0.9,
    "minimum_event_recall": 0.9,
    "targets_met": true,
    "errors": 2
  }
}
Human acceptance remains pending; no production deployment.
